# 上海交通大学 《人工智能基础与实践》 微调与数据集构建 
# 实践：使用LLaMA-Factory 进行微调

赵伟明 weiming.zhao@sjtu.edu.cn  

2025年11月

## 1. 安装 LLaMA-Factory

In [ ]:
# 1. 使用 Gitee 镜像拉取 LLaMA-Factory
!git clone --depth 1 https://gitee.com/hiyouga/LLaMA-Factory.git

In [ ]:
!pip uninstall -y vllm
!pip install llamafactory[metrics]==0.9.3

In [ ]:
## 检测是否安装成功
!llamafactory-cli version

## 2. 下载 Qwen 模型 (使用阿里内网极速下载)

In [ ]:
import os
from modelscope import snapshot_download

# 1. 指定模型：Qwen2.5-7B-Instruct
# 这是通义千问 2.5 版本的 7B 指令微调模型
# 特点：中文能力极强，指令遵循度高，非常适合作为微调的基座（不易翻车）
model_id = 'Qwen/Qwen2.5-7B-Instruct'

print(f"🚀 正在从 ModelScope 下载 {model_id}...")
print("阿里云内网速度极快 (50MB/s+)，请稍等片刻...")

# cache_dir 指定下载到当前目录下的 models 文件夹，方便好找
# revision='master' 确保下载的是主分支最新版本
model_dir = snapshot_download(model_id, cache_dir='./models', revision='master')

print(f"\n✅ 模型下载完成！路径位于: {model_dir}")

## 3. 使用代码加载“原版”模型 + 连续对话


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. 加载模型
# 注意：这里路径填刚才下载好的路径
model_path = "./models/Qwen/Qwen2___5-7B-Instruct"

print("🚀 正在加载 Qwen 原版模型...")
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    torch_dtype=torch.float16, 
    device_map="auto", 
    trust_remote_code=True
)

print("✅ 加载成功！进入对话模式 (输入 'exit' 退出)")
print("-" * 30)

# 2. 连续对话循环
# DeepSeek 有自己的对话模板，我们用官方推荐的 apply_chat_template
messages = [] # 用于存储历史记录

while True:
    query = input("\n👩‍🏫 用户: ")
    if query.strip() == "exit":
        break
    
    # 将用户问题加入历史
    messages.append({"role": "user", "content": query})
    
    # 构建 Prompt
    #inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
    inputs = tokenizer.apply_chat_template(
                        messages, 
                        return_tensors="pt", 
                        add_generation_prompt=True
                        ).to(model.device)
    
    # 生成回答
    outputs = model.generate(
        inputs, 
        max_new_tokens=512, 
        do_sample=True, 
        top_k=50, 
        top_p=0.95,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )
    
    # 解码
    response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    print(f"🤖 Qwen: {response}")
    
    # 将 AI 回答加入历史，实现连续对话
    messages.append({"role": "assistant", "content": response})

# *.可能存在的数据后处理

In [ ]:
## 可能存在的收据后处理 

import json
import re

# 1. 把你的文件名填在这里
input_file = "xiyouji4_dataset.jsonl"  # 你的原始文件
output_file = "cleaned_data.jsonl" # 清洗后的文件

cleaned_count = 0

with open(output_file, 'w', encoding='utf-8') as f_out:
    with open(input_file, 'r', encoding='utf-8') as f_in:
        for line in f_in:
            try:
                # 解析最外层
                data = json.loads(line)
                
                # 1. 捞出“问题” (在外层的 input 字段里)
                # 注意：有时候EasyDataSet会把问题放在input，有时候在instruction，视配置而定
                # 这里根据你的样本，问题在 input 里
                question = data.get('input', '')
                
                # 2. 捞出“思考过程”后面的 JSON
                raw_content = data.get('output', '')
                
                # 去掉 <think>...</think> 及其里面的废话
                # 使用非贪婪匹配，把思考部分删掉
                content_no_think = re.sub(r'<think>.*?</think>', '', raw_content, flags=re.DOTALL)
                
                # 3. 提取内层 JSON
                # 寻找第一个 { 和 最后一个 } 之间的内容
                json_match = re.search(r'\{.*\}', content_no_think, re.DOTALL)
                
                if json_match:
                    inner_json_str = json_match.group(0)
                    inner_data = json.loads(inner_json_str)
                    
                    # 4. 组装成完美的 Alpaca 格式
                    new_entry = {
                        "instruction": question,               # 问题放这里
                        "input": inner_data.get('input', ''),  # 原文背景放这里
                        "output": inner_data.get('output', '') # 答案放这里
                    }
                    
                    # 写入新文件
                    f_out.write(json.dumps(new_entry, ensure_ascii=False) + '\n')
                    cleaned_count += 1
                    
            except Exception as e:
                print(f"跳过一行坏数据: {e}")
                continue

print(f"大功告成！成功清洗并拯救了 {cleaned_count} 条数据！")
print(f"请使用新文件: {output_file} 进行微调。")

## 4. 放置你的数据集文件


1. 将数据集```json```文件放至``` LLaMA-Factory/data/``` 文件夹下
2. 在 ```LLaMA-Factory/data/dataset_info.json``` 文件的最后加入你的数据集的信息，类似
```
...
},
  "xiyouji": {
    "file_name": "xiyouji1_data.jsonl"
  }
    
}
```
一定要注意要给上一行之前加上逗号，否则JSON文件会报错。

## 5. 启动 LLaMA-Factory WebUI 进行微调

In [ ]:
!export USE_MODELSCOPE_HUB=1 && \
llamafactory-cli webui

In [ ]:
!ps

## *6. 加载微调后的模型和LoRA插件

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 1. 路径设置
# 原版模型路径 (刚才你填的那个绝对路径)
base_model_path = "./models/Qwen/Qwen2___5-7B-Instruct"

# 微调后的 LoRA 路径 (就在 LLaMA-Factory 目录下)
# 注意：请把下面的路径换成你日志里显示的那个具体路径
lora_path = "./LLaMA-Factory/saves/DeepSeek-R1-1.5B-Distill/lora/train_2025-11-23-19-40-50"

print("🚀 正在加载原版模型...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path, 
    torch_dtype=torch.float16, 
    device_map="auto", 
    trust_remote_code=True
)

print("🔌 正在挂载微调 LoRA 适配器...")
# 关键步骤：把 LoRA "挂" 到原模型上
model = PeftModel.from_pretrained(base_model, lora_path)

print("✅ 加载完成！开始测试...")

# 2. 测试对话
messages = [
    {"role": "user", "content": "你是谁？"}
]

inputs = tokenizer.apply_chat_template(
    messages, 
    return_tensors="pt", 
    add_generation_prompt=True
).to(model.device)

outputs = model.generate(
    inputs, 
    max_new_tokens=200, 
    do_sample=True, 
    temperature=0.7
)

response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print(f"\n🤖 回答: {response}")